In [1]:
# https://my-json-server.typicode.com/st8324/repo/cafe
# 카페 매출 정보를 가져와서 DataFrame으로 변환하는 코드를 작성하세요.

import requests
import pandas as pd

In [2]:
import requests
import pandas as pd

# 카페 매출 정보를 가져오는 함수
def get_cafe_info():
  """카페 매출 정보를 가져와서 DataFrame으로 변환하는 코드"""
  url = 'https://my-json-server.typicode.com/st8324/repo/cafe'
  
  response = requests.get(url)

  try:
    response.raise_for_status()
    return pd.DataFrame(response.json())
    
  except Exception as e:
    print(f'예외 발생 : {e}')
    return pd.DataFrame()
  
print(get_cafe_info())

                주문시간     메뉴 카테고리  수량    가격   결제수단
0   2026-03-01 08:30  아메리카노   커피   1  4100     카드
1   2026-03-01 12:15   카페라떼   커피   2  4600     카드
2   2026-03-01 12:45  아메리카노   커피   1  4100     쿠폰
3   2026-03-02 09:10   카푸치노   커피   1  4800     카드
4   2026-03-02 13:20   카페라떼   커피   1  4600  카카오페이
5   2026-03-02 18:00  아메리카노   커피   3  4100     카드
6   2026-03-03 11:00  자몽에이드   음료   1  5500     카드
7   2026-03-03 15:30  아메리카노   커피   2  4100     현금
8   2026-03-04 08:00  아메리카노   커피   1  4100     카드
9   2026-03-04 12:00   카페라떼   커피   2  4600     카드
10  2026-03-04 13:00   카페라떼   커피   1  4600     카드
11  2026-03-05 14:00   샌드위치   푸드   1  6500  카카오페이


In [3]:
def clean_data(df:pd.DataFrame)->pd.DataFrame:
  df = df.dropna()

  return df

In [4]:
df = get_cafe_info()

df = clean_data(df)
# print(df)

In [5]:
# 데이터를 활용한 예제
# 매출을 조회하기 위해 '결제액'을 추가
# 결제액은 가격 * 수량

df['결제액'] = df['수량'] * df['가격']

# print(df)


In [6]:
# 전체 매출액을 조회
revenue = df['결제액'].sum()

print(f"전체 매출액: {revenue:,}원")

전체 매출액: 77,200원


In [7]:
# 메뉴별 매출액을 조회(메뉴, 수량, 결제액)
df['매출액'] = df['가격'] * df['수량']


menu_df = df.groupby('메뉴')[['수량', '결제액']].sum()

print(menu_df)

       수량    결제액
메뉴              
샌드위치    1   6500
아메리카노   8  32800
자몽에이드   1   5500
카페라떼    6  27600
카푸치노    1   4800


In [8]:
# 카테고리별 수량과 결제액을 조회
category_df = df.groupby('카테고리')[['수량','결제액']].sum()
# 결제액 기준 내림차순으로 정렬
category_df = category_df.sort_values(by='결제액', ascending=False)

# 카테고리별 매출비율을 추가
# 전체 매출액 : revenue

category_df['매출비율'] = category_df['결제액'] / revenue * 100
category_df['매출비율'] = category_df['매출비율'].round(1) # 소수점 두번째 자리에서 반올림

# 매출이 가장 높은 카테고리를 조회
max_revenue = category_df['결제액'].max()
# print(category_df.loc[category_df['결제액'] == max_revenue])


# print("--------------")
max_revenue_idx = category_df['결제액'].idxmax() # 결제액 중 가장 큰 값을 가지는 위치
print(f'매출이 가장 많은 카테고리 : {max_revenue_idx}')



매출이 가장 많은 카테고리 : 커피


In [9]:
# 시간대 별 주문 건수를 조회
# 시간대를 추가
df['주문시간'] = pd.to_datetime(df['주문시간'])
df['시간'] = df['주문시간'].dt.hour

hour_df = df['시간'].value_counts()
hour_df = hour_df.sort_index(ascending=True)


# print(hour_df)

# 시간대별로 매출액을 조회 : 오전(12시 이전), 오후(17시 이전), 저녁(17시 이후)
# 시간을 시간대로 변환하는 함수
def get_time_slot(hour):
  if hour < 12:
    return "오전"
  elif hour < 17:
    return "오후"
  else:
    return "저녁"

# 시간대을 시간대로
df['시간대'] = df['시간'].apply(get_time_slot)

# 시간대 별 결제액을 조회
time_df = df.groupby('시간대')['결제액'].sum()

print(time_df)

시간대
오전    18500
오후    46400
저녁    12300
Name: 결제액, dtype: int64


In [10]:
# 요일별 결제액 조회
# dayofweek => 월 :0, 화: 1, ...., 일: 6
# day_name => 요일을 영어로
df['요일'] = df['주문시간'].dt.day_name()

# 요일을 한글로
dayweek_map= {
  'Monday' : '월요일',
  'Tuesday' : '화요일',
  'Wednesday' : '수요일',
  'Thursday' : '목요일',
  'Friday' : '금요일',
  'Saturday' : '토요일',
  'Sunday' : '일요일'
  }
df['요일'] = df['요일'].map(dayweek_map)

# 카테고리 리스트
weekday_list = ['월요일', '화요일', '수요일', '목요일', '금요일', '토요일', '일요일']

# 요일을 카테고리 타입으로 변환
df['요일'] = pd.Categorical(df['요일'], categories=weekday_list, ordered=True)

# 요일별 결제액을 조회
weekday_df = df.groupby('요일')['결제액'].sum()

# 요일 기준으로 정렬
weekday_df = weekday_df.sort_index()


print(weekday_df)


요일
월요일    21700
화요일    13700
수요일    17900
목요일     6500
금요일        0
토요일        0
일요일    17400
Name: 결제액, dtype: int64


C:\Users\C605\AppData\Local\Temp\ipykernel_7184\397625241.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  weekday_df = df.groupby('요일')['결제액'].sum()


In [13]:
# 전체를 카페_매출.csv로 저장
df.to_csv('카페_매출.csv', index=False)
# 메뉴별 매출을 카페_메뉴_매출.csv로 저장
menu_df.to_csv('카페_메뉴_매출.csv')
# 카테고리별 매출을 카페_카테고리_매출.csv로 저장
category_df.to_csv('카페_카테고리_매출.csv',encoding='utf-8-sig')
# 시간대별 매출을 카페_시간_매출.csv로 저장
time_df.to_csv('카페_시간_매출.csv', encoding='utf-8-sig')
# 요일별 매출을 카페_요일_매출.csv로 저장
weekday_df.to_csv('카페_요일_매출.csv', encoding='utf-8-sig')